In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("Metro_Interstate_Traffic_Volume.csv")

In [3]:
df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [4]:
df.shape

(48204, 9)

In [5]:
df.columns

Index(['holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main',
       'weather_description', 'date_time', 'traffic_volume'],
      dtype='object')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   holiday              61 non-null     object 
 1   temp                 48204 non-null  float64
 2   rain_1h              48204 non-null  float64
 3   snow_1h              48204 non-null  float64
 4   clouds_all           48204 non-null  int64  
 5   weather_main         48204 non-null  object 
 6   weather_description  48204 non-null  object 
 7   date_time            48204 non-null  object 
 8   traffic_volume       48204 non-null  int64  
dtypes: float64(3), int64(2), object(4)
memory usage: 3.3+ MB


In [7]:
hourly_df = (
    df.groupby("date_time", as_index=False)
      .agg({
          "temp": "mean",
          "traffic_volume": "max"
      })
)

hourly_df.head()

,date_time,temp,traffic_volume
0,2012-10-02 09:00:00,288.28,5545
1,2012-10-02 10:00:00,289.36,4516
2,2012-10-02 11:00:00,289.58,4767
3,2012-10-02 12:00:00,290.13,5026
4,2012-10-02 13:00:00,291.14,4918


In [8]:
hourly_df.shape

(40575, 3)

In [9]:
traffic = hourly_df["traffic_volume"]

mean_traffic = traffic.mean()
median_traffic = traffic.median()
std_traffic = traffic.std()
variance_traffic = traffic.var()
range_traffic = traffic.max() - traffic.min()

print("Mean:", mean_traffic)
print("Median:", median_traffic)
print("Standard deviation:", std_traffic)
print("Variance:", variance_traffic)
print("Range:", range_traffic)

Mean: 3290.650474430068
Median: 3427.0
Standard deviation: 1984.772909272217
Variance: 3939323.5013809
Range: 7280


### Interpretation of Traffic Volume Statistics

The mean hourly traffic volume is approximately 3,290.65 vehicles, meaning that an average observed hour contains about 3,291 vehicles.

The median traffic volume is 3,427 vehicles. This means that half of the hourly observations have traffic volumes below 3,427 vehicles and half have volumes above this value.

The standard deviation is approximately 1,984.77 vehicles. This is relatively large compared with the mean, indicating substantial variation in traffic volume across different hours. This is reasonable for traffic data because traffic levels can vary considerably between peak and off-peak periods.

The variance is approximately 3,939,323.50. Variance also measures the spread of traffic volume, but because it is expressed in squared units, the standard deviation is generally easier to interpret.

The range is 7,280 vehicles, showing a large difference between the lowest and highest hourly traffic volumes in the dataset.

Overall, the statistics indicate that traffic volume varies considerably across different hours rather than remaining close to a constant level.

### Task 2.2: Correlation Analysis

In [11]:
correlation = hourly_df["temp"].corr(hourly_df["traffic_volume"])

print("Correlation between temperature and traffic volume:", correlation)

Correlation between temperature and traffic volume: 0.13688415439118012


### Interpretation of Temperature and Traffic Correlation

The correlation coefficient between temperature and traffic volume is approximately **0.137**.

The positive sign indicates a **positive relationship**, meaning that higher temperatures tend to be associated with slightly higher traffic volumes.

However, the correlation is close to zero, so the relationship is **weak**. Temperature alone therefore does not appear to be a strong predictor of traffic volume.

Correlation does not imply causation. Although temperature and traffic volume may change together to some extent, this does not mean that higher temperatures directly cause higher traffic volume. Other factors such as time of day, day of the week, holidays, commuting patterns and weather conditions may also influence traffic volume.

### Task 3: Probability and Congestion Analysis

In [13]:
weather_check = (
    df.groupby("date_time")["weather_main"]
      .nunique()
)

different_weather_hours = (weather_check > 1).sum()

print(
    "Number of timestamps with more than one weather condition:",
    different_weather_hours
)

Number of timestamps with more than one weather condition: 5349


In [14]:
# Create a separate dataframe for probability analysis
prob_df = df.copy()

# Congestion: traffic volume greater than 5,500 vehicles
prob_df["congestion"] = prob_df["traffic_volume"] > 5500

# Clear weather
prob_df["clear_weather"] = prob_df["weather_main"] == "Clear"

# High temperature: greater than 292 Kelvin
prob_df["high_temperature"] = prob_df["temp"] > 292

In [15]:
print("Congestion rows:", prob_df["congestion"].sum())
print("Clear weather rows:", prob_df["clear_weather"].sum())
print("High temperature rows:", prob_df["high_temperature"].sum())
print("Total rows:", len(prob_df))

Congestion rows: 7100
Clear weather rows: 13391
High temperature rows: 11752
Total rows: 48204


In [16]:
# P(Congestion)
p_congestion = prob_df["congestion"].mean()

# P(Clear Weather)
p_clear = prob_df["clear_weather"].mean()

# P(Congestion AND Clear Weather)
p_congestion_and_clear = (
    prob_df["congestion"] & prob_df["clear_weather"]
).mean()

print("P(Congestion):", p_congestion)
print("P(Clear Weather):", p_clear)
print("P(Congestion AND Clear Weather):", p_congestion_and_clear)

P(Congestion): 0.1472906812712638
P(Clear Weather): 0.277798522944154
P(Congestion AND Clear Weather): 0.03657372832130114


### 3.1 Basic Probability

The probability of congestion is approximately **0.1473 (14.73%)**. This means that about 14.73% of the observations in the dataset have traffic volume above 5,500 vehicles.

The probability of clear weather is approximately **0.2778 (27.78%)**, meaning that about 27.78% of the weather observations are classified as clear.

The probability of both congestion and clear weather occurring together is approximately **0.0366 (3.66%)**. This means that around 3.66% of all observations contain both a traffic volume above 5,500 vehicles and clear weather conditions.

In [17]:
# Keep only congested observations
congested_df = prob_df[prob_df["congestion"]]

# P(Clear Weather | Congestion)
p_clear_given_congestion = congested_df["clear_weather"].mean()

# P(High Temperature | Congestion)
p_high_temp_given_congestion = congested_df["high_temperature"].mean()

print("P(Clear Weather | Congestion):", p_clear_given_congestion)
print("P(High Temperature | Congestion):", p_high_temp_given_congestion)

P(Clear Weather | Congestion): 0.2483098591549296
P(High Temperature | Congestion): 0.26295774647887327


### 3.2 Conditional Probability

The probability of clear weather given congestion is approximately **0.2483 (24.83%)**. This means that among all congested observations, about 24.83% occurred during clear weather conditions.

The probability of high temperature given congestion is approximately **0.2630 (26.30%)**. This means that among all congested observations, about 26.30% occurred when the temperature was above 292 K.

These results suggest that congestion occurs under a range of weather and temperature conditions rather than being concentrated only during clear or high-temperature periods.

In [18]:
expected_if_independent = p_congestion * p_clear

print("P(Congestion AND Clear Weather):", p_congestion_and_clear)
print("P(Congestion) × P(Clear Weather):", expected_if_independent)
print("Difference:", p_congestion_and_clear - expected_if_independent)

P(Congestion AND Clear Weather): 0.03657372832130114
P(Congestion) × P(Clear Weather): 0.04091713370059525
Difference: -0.004343405379294112


### Independence of Congestion and Clear Weather

To examine whether congestion and clear weather are independent, the observed joint probability was compared with the probability expected under independence.

- P(Congestion AND Clear Weather) = **0.0366 (3.66%)**
- P(Congestion) × P(Clear Weather) = **0.0409 (4.09%)**

The two values are not equal. Therefore, congestion and clear weather cannot be considered perfectly independent in this dataset.

The observed joint probability is slightly lower than the value expected under independence. This suggests that clear weather and congestion may have a weak association. However, the difference is relatively small, so weather alone is unlikely to explain congestion patterns.

In [19]:
# Keep only Clear and Clouds observations
clear_cloudy_df = prob_df[
    prob_df["weather_main"].isin(["Clear", "Clouds"])
]

# Build a 2x2 contingency table
odds_table = pd.crosstab(
    clear_cloudy_df["weather_main"],
    clear_cloudy_df["congestion"]
)

odds_table

congestion,False,True
weather_main,,
Clear,11628,1763
Clouds,12572,2592


In [20]:
clear_congested = odds_table.loc["Clear", True]
clear_not_congested = odds_table.loc["Clear", False]

clouds_congested = odds_table.loc["Clouds", True]
clouds_not_congested = odds_table.loc["Clouds", False]

clear_odds = clear_congested / clear_not_congested
clouds_odds = clouds_congested / clouds_not_congested

odds_ratio = clear_odds / clouds_odds

print("Clear weather congestion odds:", clear_odds)
print("Cloudy weather congestion odds:", clouds_odds)
print("Odds ratio:", odds_ratio)

Clear weather congestion odds: 0.15161678706570347
Cloudy weather congestion odds: 0.20617244670696785
Odds ratio: 0.7353882125733118


### Odds Ratio: Clear vs Cloudy Weather

The odds ratio of congestion in clear weather compared with cloudy weather is approximately **0.735**.

An odds ratio below 1 indicates that congestion has lower odds during clear weather than during cloudy weather. In this dataset, the odds of congestion during clear weather are approximately **26.5% lower** than during cloudy weather.

However, this result does not demonstrate that cloudy weather causes congestion. Traffic conditions may also be influenced by factors such as time of day, commuting patterns, day of the week, holidays and other weather conditions.

### Overall Probability Analysis Conclusion

Congestion occurred in approximately **14.73%** of the observations, while clear weather occurred in approximately **27.78%**. About **3.66%** of all observations experienced both congestion and clear weather.

Among congested observations, approximately **24.83%** occurred during clear weather and **26.30%** occurred when temperatures exceeded 292 K.

The independence analysis showed that the observed probability of congestion and clear weather occurring together (**3.66%**) was slightly lower than the probability expected if the two events were independent (**4.09%**). This suggests a small association between weather conditions and congestion.

The odds-ratio analysis also showed that congestion had lower odds during clear weather than during cloudy weather. Overall, weather appears to have some relationship with congestion, but the results suggest that weather alone is unlikely to explain traffic congestion.